# Proyecto Integrador: Predictor de demanda para redes eléctricas
## Ingeniería de Datos - Arquitectura Medallón (Capas Bronce, Plata y Oro)
**Curso de Especialización en IA y Big Data - IES San Andrés**

Este cuaderno contiene el pipeline central de procesamiento utilizando **Apache Spark** y el ecosistema **AWS**. Aquí se documenta e implementa el flujo de datos completo: la ingesta y almacenamiento en crudo (Capa Bronce), la limpieza y normalización de formatos (Capa Plata), y el cruce final con *Feature Engineering* (Capa Oro) para generar el dataset predictivo maestro.

## 0.Configuración inicial y Test

In [1]:
!pip install boto3


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import boto3
from pyspark.sql import SparkSession

# Test AWS
try:
    s3 = boto3.client('s3')
    print("Conexión a AWS S3 exitosa.")
except Exception as e:
    print("Error en AWS:", e)

# Iniciar Spark
spark = (SparkSession.builder
         .appName("Proyecto_Integrador_Demanda")
         .master("spark://spark-master:7077")
         .getOrCreate()
        )
print("SparkSession iniciada correctamente.")

Conexión a AWS S3 exitosa.


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/22 15:08:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession iniciada correctamente.


In [3]:
respuesta= s3.list_buckets()

# Recorremos la lista de diccionarios que nos da AWS
for bucket in respuesta['Buckets']:
    nombre = bucket['Name']
    print(f"Nombre del bucket: {nombre}")

Nombre del bucket: aws-logs-211125762907-us-east-1
Nombre del bucket: cag-bd-iessanandres-practicas-ut6
Nombre del bucket: cag-iessanandres-bd-ia
Nombre del bucket: cag-proyecto-demanda-bd
Nombre del bucket: cag-proyecto-demanda-electrica-bd
Nombre del bucket: smartagro-raw-data-cag


26/05/22 15:08:29 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [8]:
BUCKET_NAME = "cag-proyecto-demanda-electrica-bd"
s3.create_bucket(Bucket=BUCKET_NAME)

{'ResponseMetadata': {'RequestId': 'N2V06XKDP4EYSHQD',
  'HostId': 'kyoFZ5/86YPR8ZwYwBZhKEXlCeu2yq4ijxI/iALrb2NXEyTbeLMk1zW95Rnl/MNS1PeLotaUVII=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'kyoFZ5/86YPR8ZwYwBZhKEXlCeu2yq4ijxI/iALrb2NXEyTbeLMk1zW95Rnl/MNS1PeLotaUVII=',
   'x-amz-request-id': 'N2V06XKDP4EYSHQD',
   'date': 'Tue, 19 May 2026 17:55:34 GMT',
   'location': '/cag-proyecto-demanda-electrica-bd',
   'x-amz-bucket-arn': 'arn:aws:s3:::cag-proyecto-demanda-electrica-bd',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'Location': '/cag-proyecto-demanda-electrica-bd',
 'BucketArn': 'arn:aws:s3:::cag-proyecto-demanda-electrica-bd'}

In [11]:
!apt-get install -y --no-install-recommends unar

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  binutils binutils-common binutils-x86-64-linux-gnu bzip2 dpkg dpkg-dev
  fontconfig gnustep-base-common gnustep-base-runtime gnustep-common
  gnustep-multiarch gnutls-bin graphviz libabsl20240722 libann0 libaom3
  libavahi-client3 libavahi-common-data libavahi-common3 libavif16 libbinutils
  libcdt5 libcgraph6 libctf-nobfd0 libctf0 libdatrie1 libdav1d7 libdbus-1-3
  libde265-0 libdeflate0 libdpkg-perl libevent-2.1-7t64 libfribidi0 libgav1-1
  libgc1 libgcrypt20 libgd3 libgnustep-base1.31 libgnutls-dane0t64
  libgnutls30t64 libgomp1 libgpg-error0 libgprofng0 libgts-0.7-5t64 libgvc6
  libgvpr2 libheif-plugin-dav1d libheif-plugin-libde265 libheif1
  libimagequant0 libjansson4 libjbig0 liblab-gamut1 liblerc4 libltdl7 libobjc4
  libpango-1.0-0 libpangocairo-1.0-0 libpangoft2-1.0-0 libpathplan4
  librav1e0.7 libsframe1 libsharpyuv0 libsvtav1e

In [6]:
!pip install pandas


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## 1. Capa Bronce - Carga Historica -> S3

### 1.1 Fuente ESIOS

In [9]:
import json
import requests
import boto3
from botocore.exceptions import ClientError
from datetime import datetime, timedelta

# --- CONFIGURACIÓN S3 ---
NOMBRE_BUCKET = BUCKET_NAME
s3 = boto3.client('s3')

def existe_en_s3(bucket, key):
    """
    Comprueba si un archivo ya existe en S3 para evitar descargas duplicadas.
    """
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError:
        return False

def guardar_mes_s3(mes_id, datos_mes):
    """
    Sube el JSON del mes directamente a S3 aplicando la arquitectura Medallón.
    """
    year, month = mes_id.split("-")
    s3_key = f"bronce/demanda/year={year}/month={month}/{mes_id}.json"
    
    s3.put_object(
        Bucket=NOMBRE_BUCKET,
        Key=s3_key,
        Body=json.dumps(datos_mes, ensure_ascii=False, indent=2),
        ContentType='application/json'
    )
    return s3_key

def obtener_demanda_json(inicio, fin):
    """
    Extrae los datos de la API de ESIOS.
    """
    url = "https://apidatos.ree.es/es/datos/balance/balance-electrico"
    params = {
        "start_date": inicio,
        "end_date": fin,
        "time_trunc": "day"
    }

    r = requests.get(url, params=params)
    if r.status_code != 200:
        print(f"Error en la API (Código {r.status_code})")
        return []
        
    data = r.json()

    for grupo in data.get("included", []):
        if grupo.get("type") == "Demanda":
            for sub in grupo["attributes"].get("content", []):
                if sub.get("type") == "Demanda en b.c.":
                    valores = sub["attributes"]["values"]
                    return [
                        {"datetime": v["datetime"], "value": float(v["value"])}
                        for v in valores
                    ]
    return []

def descargar_demanda_json_desde_2013():
    """
    Itera desde 2013 hasta la actualidad, comprobando y subiendo datos a S3.
    """
    fecha_inicio = datetime(2013, 1, 1)
    fecha_fin = datetime.now()
    fecha_actual = fecha_inicio

    nuevos_descargados = 0

    while fecha_actual < fecha_fin:
        inicio = fecha_actual.strftime("%Y-%m-%dT00:00")
        fin_mes = (fecha_actual + timedelta(days=32)).replace(day=1) - timedelta(seconds=1)
        fin = min(fin_mes, fecha_fin).strftime("%Y-%m-%dT23:59")
        mes_id = inicio[:7]  # Formato: YYYY-MM

        # Definimos la ruta que tendrá en S3 para comprobar si ya existe
        year, month = mes_id.split("-")
        s3_key = f"bronce/demanda/year={year}/month={month}/{mes_id}.json"
        
        if existe_en_s3(NOMBRE_BUCKET, s3_key):
            print(f"Omitiendo demanda de {mes_id} (Ya existe en S3)")
        else:
            print(f"Descargando demanda de {mes_id}...")
            datos_mes = obtener_demanda_json(inicio, fin)
            
            if datos_mes:
                guardar_mes_s3(mes_id, datos_mes)
                nuevos_descargados += 1

        # Avanzar al siguiente mes
        fecha_actual = (fecha_actual + timedelta(days=32)).replace(day=1)
        
    print(f"\nProceso histórico finalizado. Se subieron {nuevos_descargados} meses nuevos a la Capa Bronce.")

# --- EJECUCIÓN ---
if __name__ == "__main__":
    descargar_demanda_json_desde_2013()

Descargando demanda de 2013-01...
Descargando demanda de 2013-02...
Descargando demanda de 2013-03...
Descargando demanda de 2013-04...
Descargando demanda de 2013-05...
Descargando demanda de 2013-06...
Descargando demanda de 2013-07...
Descargando demanda de 2013-08...
Descargando demanda de 2013-09...
Descargando demanda de 2013-10...
Descargando demanda de 2013-11...
Descargando demanda de 2013-12...
Descargando demanda de 2014-01...
Descargando demanda de 2014-02...
Descargando demanda de 2014-03...
Descargando demanda de 2014-04...
Descargando demanda de 2014-05...
Descargando demanda de 2014-06...
Descargando demanda de 2014-07...
Descargando demanda de 2014-08...
Descargando demanda de 2014-09...
Descargando demanda de 2014-10...
Descargando demanda de 2014-11...
Descargando demanda de 2014-12...
Descargando demanda de 2015-01...
Descargando demanda de 2015-02...
Descargando demanda de 2015-03...
Descargando demanda de 2015-04...
Descargando demanda de 2015-05...
Descargando de

### 1.2 Fuente histórico de clima

In [16]:
import os
import requests
import subprocess
import shutil
import boto3
import calendar
from bs4 import BeautifulSoup
from botocore.exceptions import ClientError

# --- 1. CONFIGURACIÓN ---
URL = "https://datosclima.es/Aemet2013/DescargaDatos.html"
CARPETA_RAR = "aemet_rar"
CARPETA_XLS = "aemet_xls"
NOMBRE_BUCKET = BUCKET_NAME

# Crear carpetas temporales en tu disco duro si no existen
os.makedirs(CARPETA_RAR, exist_ok=True)
os.makedirs(CARPETA_XLS, exist_ok=True)

# Cliente S3
s3 = boto3.client('s3')

# --- 2. FUNCIONES AUXILIARES ---
def archivo_existe_en_s3(bucket, key):
    """Comprueba si el archivo ya está en AWS para no subirlo dos veces."""
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError:
        return False

def obtener_links_rar():
    """Hace web scraping para sacar todos los links de descarga .rar"""
    print("Buscando archivos .rar en la web...")
    html = requests.get(URL).text
    soup = BeautifulSoup(html, "html.parser")
    links = []
    
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".rar"):
            links.append("https://datosclima.es/Aemet2013/" + href)
            
    print(f"   -> Encontrados {len(links)} meses históricos.")
    return links

def descargar_y_extraer_rars(links):
    """Descarga los RAR, repara meses incompletos calculando los días exactos y extrae los XLS."""
    print("Descargando y extrayendo (lógica de calendario exacto activada)...")
    
    for i, url in enumerate(links):
        es_ultimo_mes = (i == len(links) - 1)
        
        nombre_rar = url.split("/")[-1]
        ruta_rar = os.path.join(CARPETA_RAR, nombre_rar)
        nombre_mes = nombre_rar.replace(".rar", "") # Ejemplo: "Aemet2024-05"
        carpeta_mes = os.path.join(CARPETA_XLS, nombre_mes)
        
        # Averiguar cuántos días DEBERÍA tener este mes exactamente
        try:
            partes = nombre_mes.replace("Aemet", "").split("-")
            year_int = int(partes[0])
            month_int = int(partes[1])
            dias_esperados = calendar.monthrange(year_int, month_int)[1]
        except Exception:
            dias_esperados = 31 # Seguro por si hay nombres raros
        
        # Contar cuántos Excel hay realmente descargados en local
        cantidad_xls = 0
        if os.path.exists(carpeta_mes):
            cantidad_xls = len([f for f in os.listdir(carpeta_mes) if f.endswith('.xls')])
        
        # Lógica anti-fallos: Si no es el mes actual y ya tiene los días exactos, saltamos.
        if not es_ultimo_mes and cantidad_xls >= dias_esperados:
             continue
             
        # Mensajes de consola
        if es_ultimo_mes:
            print(f" -> Actualizando mes en curso: {nombre_mes}")
        else:
            print(f" -> Descargando/Reparando: {nombre_mes} (Tiene {cantidad_xls} / Faltan hasta {dias_esperados})")
             
        # Descarga del RAR
        r = requests.get(url)
        if r.status_code == 200:
            with open(ruta_rar, "wb") as f:
                f.write(r.content)
        else:
            print(f"Error al descargar {nombre_rar} de la web.")
            continue
            
        # Extracción forzada con unar
        os.makedirs(carpeta_mes, exist_ok=True)
        subprocess.run(["unar", "-f", "-o", carpeta_mes, ruta_rar], check=False, stdout=subprocess.DEVNULL)
        
        # Limpieza: Mover XLS y eliminar subcarpetas dobles
        for root, dirs, files in os.walk(carpeta_mes):
            for f in files:
                if f.lower().endswith(".xls"):
                    origen = os.path.join(root, f)
                    destino = os.path.join(carpeta_mes, f)
                    if origen != destino:
                        shutil.move(origen, destino)

def subir_xls_a_s3():
    """Sube la estructura de carpetas a Amazon S3 respetando la Capa Bronce."""
    print("Subiendo archivos XLS a Amazon S3...")
    archivos_subidos = 0
    
    for root, dirs, files in os.walk(CARPETA_XLS):
        for file in files:
            if file.lower().endswith(".xls"):
                ruta_local = os.path.join(root, file)
                
                # Extraer año y mes. Ejemplo de file: "Aemet2024-05-02.xls"
                nombre_sin_ext = file.replace(".xls", "").replace(".XLS", "")
                partes = nombre_sin_ext.split("-")
                
                try:
                    year = partes[0][-4:]
                    month = partes[1]
                except IndexError:
                    continue 
                
                # Definir ruta destino en S3
                s3_key = f"bronce/clima/year={year}/month={month}/{file}"
                
                # Subir solo si no existe ya en S3
                if not archivo_existe_en_s3(NOMBRE_BUCKET, s3_key):
                    s3.upload_file(ruta_local, NOMBRE_BUCKET, s3_key)
                    archivos_subidos += 1
                    
    print(f"\nProceso histórico finalizado. Se subieron {archivos_subidos} archivos XLS nuevos a S3.")

# --- 3. EJECUCIÓN PRINCIPAL ---
if __name__ == "__main__":
    links_a_descargar = obtener_links_rar()
    descargar_y_extraer_rars(links_a_descargar)
    subir_xls_a_s3()

Buscando archivos .rar en la web...
   -> Encontrados 155 meses históricos.
Descargando y extrayendo (lógica de calendario exacto activada)...
 -> Descargando/Reparando: Aemet2013-05 (Tiene 25 / Faltan hasta 31)
 -> Descargando/Reparando: Aemet2013-11 (Tiene 29 / Faltan hasta 30)
 -> Descargando/Reparando: Aemet2014-04 (Tiene 29 / Faltan hasta 30)
 -> Descargando/Reparando: Aemet2014-09 (Tiene 29 / Faltan hasta 30)
 -> Descargando/Reparando: Aemet2016-07 (Tiene 30 / Faltan hasta 31)
 -> Descargando/Reparando: Aemet2017-02 (Tiene 27 / Faltan hasta 28)
 -> Descargando/Reparando: Aemet2020-03 (Tiene 28 / Faltan hasta 31)
 -> Actualizando mes en curso: Aemet2026-03
Subiendo archivos XLS a Amazon S3...

Proceso histórico finalizado. Se subieron 4159 archivos XLS nuevos a S3.


### 1.3 Fuente de Festivos

In [21]:
import requests
import json
import boto3

# --- 1.CONFIGURACIÓN ---
API_KEY = "Xb162V3SZvCnbe7DKGgVvPVhn8kQNAFX"
COUNTRY = "ES"
NOMBRE_BUCKET = BUCKET_NAME

# Cliente S3
s3 = boto3.client('s3')

# --- 2. FUNCIONES AUXILIARES ---
def archivo_existe_en_s3(bucket, key):
    """Comprueba si el archivo ya está en AWS para no repetir trabajo."""
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError:
        return False

def get_festivos(year):
    """Llama a la API y devuelve los datos en la memoria RAM."""
    url = "https://calendarific.com/api/v2/holidays"
    params = {
        "api_key": API_KEY,
        "country": COUNTRY,
        "year": year
    }
    resp = requests.get(url, params=params)
    if resp.status_code == 200:
        return resp.json()
    else:
        print(f"Error en API para el año {year}: {resp.status_code}")
        return None

def ingesta_directa_s3():
    print("Iniciando ingesta a S3...")
    archivos_subidos = 0
    
    for year in range(2013, 2027):
        # Definir ruta destino particionada en S3
        s3_key = f"bronce/festivos/year={year}/festivos_{year}.json"
        
        # 1. Comprobar S3 antes de gastar llamada a la API
        if archivo_existe_en_s3(NOMBRE_BUCKET, s3_key):
            print(f"-> Omitiendo {year} (Ya existe en S3)")
            continue
            
        # 2. Descargar a la memoria RAM
        print(f"-> Extrayendo año {year} de Calendarific...")
        data = get_festivos(year)
        
        # 3. Inyectar en S3 directamente desde la variable 'data'
        if data:
            s3.put_object(
                Bucket=NOMBRE_BUCKET,
                Key=s3_key,
                Body=json.dumps(data, ensure_ascii=False, indent=2),
                ContentType='application/json'
            )
            print(f" Subido a S3: {s3_key}")
            archivos_subidos += 1
            
    print(f"\nProceso histórico finalizado. Se subieron {archivos_subidos} años nuevos a S3.")

# --- 3. EJECUCIÓN PRINCIPAL ---
if __name__ == "__main__":
    ingesta_directa_s3()

Iniciando ingesta a S3...
-> Omitiendo 2013 (Ya existe en S3)
-> Omitiendo 2014 (Ya existe en S3)
-> Omitiendo 2015 (Ya existe en S3)
-> Omitiendo 2016 (Ya existe en S3)
-> Omitiendo 2017 (Ya existe en S3)
-> Omitiendo 2018 (Ya existe en S3)
-> Omitiendo 2019 (Ya existe en S3)
-> Omitiendo 2020 (Ya existe en S3)
-> Omitiendo 2021 (Ya existe en S3)
-> Omitiendo 2022 (Ya existe en S3)
-> Omitiendo 2023 (Ya existe en S3)
-> Omitiendo 2024 (Ya existe en S3)
-> Omitiendo 2025 (Ya existe en S3)
-> Omitiendo 2026 (Ya existe en S3)

Proceso histórico finalizado. Se subieron 0 años nuevos a S3.


## 2. Capa plata: Limpieza de datos

### 2.2 Fuente ESIOS 

In [5]:
import os
import shutil
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, round

# --- CONFIGURACIÓN ---
BUCKET_NAME = "cag-proyecto-demanda-electrica-bd"
CARPETA_TEMP = "descargas_bronce_demanda"
CARPETA_PLATA = "plata/demanda_esios"
s3 = boto3.client('s3')

def procesar_plata_demanda():
    print("=== INICIANDO CAPA PLATA: DEMANDA ELÉCTRICA ===")
    if os.path.exists(CARPETA_TEMP): shutil.rmtree(CARPETA_TEMP)
    os.makedirs(CARPETA_TEMP, exist_ok=True)
    
    print("-> Descargando JSONs desde S3...")
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="bronce/demanda/"):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".json"):
                s3.download_file(BUCKET_NAME, obj["Key"], os.path.join(CARPETA_TEMP, obj["Key"].split("/")[-1]))

    print("-> Limpiando con Apache Spark...")
    spark = SparkSession.builder.appName("Plata_Demanda").master("local[*]").getOrCreate()
    
    try:
        # Lectura multilínea activada para tu JSON
        df_crudo = spark.read.option("multiline", "true").json(f"{CARPETA_TEMP}/*.json")
        
        # Como es diario, solo sacamos fecha y megavatios totales
        df_limpio = df_crudo.select(
            to_date(col("datetime")).alias("fecha"),
            round(col("value").cast("float"), 2).alias("megavatios_dia")
        ).dropDuplicates()

        print(f"-> Guardando Parquet en '{CARPETA_PLATA}'...")
        df_limpio.coalesce(1).write.mode("overwrite").parquet(CARPETA_PLATA)
        print(f"¡Éxito! Registros guardados: {df_limpio.count()}")

    except Exception as e:
        print(f"Error: {e}")
    finally:
        spark.stop()
        if os.path.exists(CARPETA_TEMP): shutil.rmtree(CARPETA_TEMP)

if __name__ == "__main__":
    procesar_plata_demanda()

=== INICIANDO CAPA PLATA: DEMANDA ELÉCTRICA ===
-> Descargando JSONs desde S3...
-> Limpiando con Apache Spark...


-> Guardando Parquet en 'plata/demanda_esios'...


¡Éxito! Registros guardados: 4887


#### 2.1.1 Subida a S3

In [7]:
import os
import boto3

# --- CONFIGURACIÓN ---
BUCKET_NAME = "cag-proyecto-demanda-electrica-bd"
CARPETA_LOCAL_PLATA = "plata/demanda_esios"  
PREFIX_S3 = "plata/demanda/"                  

def subir_plata_a_s3():
    print("=== SUBIENDO CAPA PLATA A AMAZON S3 ===")
    s3 = boto3.client('s3')
    
    if not os.path.exists(CARPETA_LOCAL_PLATA):
        print(f"No se encuentra la carpeta local {CARPETA_LOCAL_PLATA}")
        return

    archivos_subidos = 0
    # Recorremos los archivos que ha creado Spark (.parquet, _SUCCESS, etc.)
    for root, dirs, files in os.walk(CARPETA_LOCAL_PLATA):
        for file in files:
            if file.endswith(".parquet") and not file.startswith("."):
                ruta_local = os.path.join(root, file)
            
                # Creamos la ruta de destino en S3
                ruta_s3 = os.path.join(PREFIX_S3, "demanda_limpia.parquet").replace("\\", "/")
                
                print(f"-> Subiendo {file} a S3...")
                s3.upload_file(ruta_local, BUCKET_NAME, ruta_s3)
                archivos_subidos += 1

    print(f"¡Capa Plata respaldada en S3 con éxito! Se han subido {archivos_subidos} archivos.")

if __name__ == "__main__":
    subir_plata_a_s3()

=== SUBIENDO CAPA PLATA A AMAZON S3 ===
-> Subiendo part-00000-4fab0318-47df-4b13-9776-67ba2a4522f3-c000.snappy.parquet a S3...
¡Capa Plata respaldada en S3 con éxito! Se han subido 1 archivos.


### 2.2 Fuente historico clima

In [8]:
import os
import shutil
import boto3
import io
import re
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, to_date, avg, round

# --- 1. CONFIGURACIÓN ---
BUCKET_NAME = "cag-proyecto-demanda-electrica-bd"
CARPETA_TEMP = "descargas_bronce_clima"
CARPETA_PLATA = "plata/clima_final"

s3 = boto3.client('s3')

# --- 2. FUNCIONES ---
def preparar_carpetas():
    os.makedirs(CARPETA_TEMP, exist_ok=True)

def descargar_bronce():
    print("-> Descargando Excels de AEMET desde S3 (Modo Resumible)...")
    paginator = s3.get_paginator('list_objects_v2')
    archivos_nuevos = 0
    archivos_cache = 0
    
    for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="bronce/clima/"):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".xls") or obj["Key"].endswith(".xlsx"):
                nombre_archivo = obj["Key"].split("/")[-1]
                ruta_local = os.path.join(CARPETA_TEMP, nombre_archivo)
                
                if os.path.exists(ruta_local):
                    archivos_cache += 1
                    continue
                try:
                    s3.download_file(BUCKET_NAME, obj["Key"], ruta_local)
                    archivos_nuevos += 1
                except Exception as e:
                    print(f"Error temporal descargando {nombre_archivo}: {e}")
                    
    print(f"-> Descarga lista. Nuevos: {archivos_nuevos} | En caché: {archivos_cache}")

# --- Función para procesar binarios en paralelo ---
def procesar_xls_binario(datos_binarios):
    import io
    import pandas as pd
    import re
    ruta, contenido = datos_binarios
    try:
        # Extraemos la fecha del nombre del archivo (ej: Aemet2013-05-07.xls)
        match = re.search(r'(\d{4}-\d{2}-\d{2})', ruta)
        fecha_str = match.group(1) if match else None

        # Leemos el binario directamente en memoria saltando las cabeceras
        pdf = pd.read_excel(io.BytesIO(contenido), engine="xlrd", skiprows=4)
        
        pdf.rename(columns={
            'Estación': 'estacion',
            'Temperatura máxima (ºC)': 'temp_max',
            'Temperatura mínima (ºC)': 'temp_min',
            'Temperatura media (ºC)': 'temp_media',
            'Precipitación 00-24h (mm)': 'precipitacion_mm'
        }, inplace=True)

        cols = ['temp_max', 'temp_min', 'temp_media', 'precipitacion_mm']
        pdf = pdf[[c for c in cols if c in pdf.columns]]
        pdf['fecha_str'] = fecha_str
        
        # Convertimos todo a texto para que Spark no se queje de tipos mixtos
        return pdf.dropna(how='all', subset=cols).astype(str).to_dict("records")
    except Exception:
        return []

def transformar_y_guardar():
    print("-> Iniciando Spark con procesamiento distribuido de archivos...")
    spark = (
        SparkSession.builder
        .appName("Plata_Clima")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.executor.memory", "4g")
        .getOrCreate()
    )
    
    try:
        rutas_archivos = f"file://{os.path.abspath(CARPETA_TEMP)}/*.xls*"
        
        print("-> Analizando Excels y limpiando textos sucios...")
        df_clima_raw = (
            spark.sparkContext
            .binaryFiles(rutas_archivos)
            .flatMap(procesar_xls_binario)
            .toDF()
        )
        
        if df_clima_raw.isEmpty():
            print("No se pudieron cargar los datos.")
            return

         #Limpieza de 'nan' y textos para matemáticas ---
        df_limpio_textos = (
            df_clima_raw
            .replace("nan", None)
            .withColumn("fecha", to_date(col("fecha_str")))
            .withColumn("temp_max_num", split(col("temp_max"), " ")[0].cast("float"))
            .withColumn("temp_min_num", split(col("temp_min"), " ")[0].cast("float"))
            .withColumn("temp_media_num", split(col("temp_media"), " ")[0].cast("float"))
            .withColumn("precipitacion_num", split(col("precipitacion_mm"), " ")[0].cast("float"))
        )

        print("-> Calculando media NACIONAL para cruzar con demanda eléctrica...")
        # Lógica de negocio
        df_final = df_limpio_textos.groupBy("fecha").agg(
            round(avg("temp_max_num"), 2).alias("temp_max_media"),
            round(avg("temp_min_num"), 2).alias("temp_min_media"),
            round(avg("temp_media_num"), 2).alias("temp_media_nacional"),
            round(avg("precipitacion_num"), 2).alias("precipitacion_media")
        ).dropna(subset=["fecha"]).orderBy("fecha")

        print(f"-> Guardando en '{CARPETA_PLATA}'...")
        if os.path.exists(CARPETA_PLATA):
            shutil.rmtree(CARPETA_PLATA)
            
        df_final.coalesce(1).write.mode("overwrite").parquet(CARPETA_PLATA)
        print(f"¡Éxito! Clima procesado: {df_final.count()} días únicos.")
        
        df_final.show(5)

    except Exception as e:
        print(f"Error crítico: {e}")
    finally:
        spark.stop()

def ejecutar_plata_clima():
    print("=== INICIANDO CAPA PLATA: CLIMA (VERSIÓN DISTRIBUIDA) ===")
    preparar_carpetas()
    descargar_bronce()
    transformar_y_guardar()

if __name__ == "__main__":
    ejecutar_plata_clima()

=== INICIANDO CAPA PLATA: CLIMA (VERSIÓN DISTRIBUIDA) ===
-> Descargando Excels de AEMET desde S3 (Modo Resumible)...
-> Descarga lista. Nuevos: 0 | En caché: 4704
-> Iniciando Spark con procesamiento distribuido de archivos...
-> Analizando Excels y limpiando textos sucios...


26/05/21 21:46:46 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 0 (TID 0): Attempting to kill Python Worker
26/05/21 21:46:50 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 1 (TID 1): Attempting to kill Python Worker
                                                                                

-> Calculando media NACIONAL para cruzar con demanda eléctrica...
-> Guardando en 'plata/clima_final'...


WARNING *** File is truncated, or OLE2 MSAT is corrupt!!=====>      (8 + 1) / 9]
INFO: Trying to access sector 520 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 511 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 520 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 530 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 520 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 520 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 534 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 514 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!=====>  

¡Éxito! Clima procesado: 4681 días únicos.


WARNING *** File is truncated, or OLE2 MSAT is corrupt!!=====>      (8 + 1) / 9]
INFO: Trying to access sector 520 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 511 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 520 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 530 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 520 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 520 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 534 but only 511 available
WARNING *** File is truncated, or OLE2 MSAT is corrupt!!
INFO: Trying to access sector 514 but only 511 available
                                                                

+----------+--------------+--------------+-------------------+-------------------+
|     fecha|temp_max_media|temp_min_media|temp_media_nacional|precipitacion_media|
+----------+--------------+--------------+-------------------+-------------------+
|2013-05-07|         23.62|         13.05|              18.34|               1.06|
|2013-05-08|         23.31|         13.42|              18.37|               0.57|
|2013-05-09|         22.01|         12.66|              17.34|               2.95|
|2013-05-10|         21.77|         11.29|              16.53|               0.13|
|2013-05-11|         21.47|          9.44|              15.46|               0.03|
+----------+--------------+--------------+-------------------+-------------------+
only showing top 5 rows



#### 2.2.1 Subida a S3 

In [9]:
import os
import boto3

# --- CONFIGURACIÓN ---
BUCKET_NAME = "cag-proyecto-demanda-electrica-bd"
CARPETA_LOCAL_CLIMA = "plata/clima_final"
PREFIX_S3 = "plata/clima" 

s3 = boto3.client('s3')

def subir_clima_s3():
    print("=== INICIANDO SUBIDA DE CLIMA A S3 ===")
    archivos_subidos = 0
    
    # Escaneamos la carpeta local donde Spark guardó el Parquet
    for root, dirs, files in os.walk(CARPETA_LOCAL_CLIMA):
        for file in files:
            # Filtramos para coger solo el archivo de datos real
            if file.endswith(".parquet") and not file.startswith("."):
                ruta_local = os.path.join(root, file)
                
                # Construimos la ruta en S3 forzando el nombre limpio y estático
                ruta_s3 = os.path.join(PREFIX_S3, "clima_limpio.parquet").replace("\\", "/")
                
                print(f"-> Subiendo {file} a S3 como '{ruta_s3}'...")
                s3.upload_file(ruta_local, BUCKET_NAME, ruta_s3)
                archivos_subidos += 1
                
    if archivos_subidos > 0:
        print(f"¡Éxito! El archivo de clima ya está seguro en S3.")
    else:
        print("No se encontró ningún archivo Parquet válido para subir.")

if __name__ == "__main__":
    subir_clima_s3()

=== INICIANDO SUBIDA DE CLIMA A S3 ===
-> Subiendo part-00000-85f250e2-b96b-4977-85a9-ca32b65c47e1-c000.snappy.parquet a S3 como 'plata/clima/clima_limpio.parquet'...
¡Éxito! El archivo de clima ya está seguro en S3.


### 2.3 Fuente festivos 

In [11]:
import os
import sys
import json
import shutil
import boto3
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

# --- 1. CONFIGURACIÓN ---
BUCKET_NAME = "cag-proyecto-demanda-electrica-bd"
CARPETA_TEMP = "descargas_bronce_festivos"
CARPETA_PLATA = "plata/festivos"

s3 = boto3.client('s3')

# --- 2. FUNCIONES ---
def preparar_carpetas():
    # Mantenemos la carpeta para permitir la reanudación si hay cortes
    os.makedirs(CARPETA_TEMP, exist_ok=True)

def descargar_bronze():
    print("-> Descargando JSONs de Festivos desde S3 (Modo Resumible)...")
    paginator = s3.get_paginator('list_objects_v2')
    archivos_nuevos = 0
    archivos_cache = 0
    
    for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="bronce/festivos/"):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".json"):
                nombre_archivo = obj["Key"].split("/")[-1]
                ruta_local = os.path.join(CARPETA_TEMP, nombre_archivo)
                
                # Tolerancia a fallos: saltar si ya existe
                if os.path.exists(ruta_local):
                    archivos_cache += 1
                    continue
                try:
                    s3.download_file(BUCKET_NAME, obj["Key"], ruta_local)
                    archivos_nuevos += 1
                except Exception as e:
                    print(f"Error temporal descargando {nombre_archivo}: {e}")
                    
    print(f"-> Descarga lista. Nuevos: {archivos_nuevos} | En caché: {archivos_cache}")

def transformar_y_guardar():
    print("-> Leyendo JSONs y aplanando datos con Pandas...")
    lista_df = []
    
    for archivo in os.listdir(CARPETA_TEMP):
        if archivo.endswith(".json"):
            ruta_completa = os.path.join(CARPETA_TEMP, archivo)
            with open(ruta_completa, 'r', encoding='utf-8') as f:
                try:
                    datos = json.load(f)
                    holidays = datos.get("response", {}).get("holidays", [])
                    if holidays:
                        df = pd.json_normalize(holidays)
                        lista_df.append(df)
                except Exception as e:
                    print(f"Omitiendo {archivo}: {e}")

    if not lista_df:
        print("No se encontraron datos válidos.")
        return

    df_total = pd.concat(lista_df, ignore_index=True)
    
    print("-> Aplicando filtros de negocio (Solo Festivos Reales)...")
    tipos_validos = ["National holiday", "Autonomous Community Holiday", "Common local holiday"]
    df_filtrado = df_total[df_total['primary_type'].isin(tipos_validos)].copy()
    
    # Limpiamos la fecha y las columnas
    df_filtrado['fecha_cruda'] = df_filtrado['date.iso'].astype(str).str[:10]
    df_filtrado.rename(columns={'name': 'festividad'}, inplace=True)
    df_final_pandas = df_filtrado[['fecha_cruda', 'festividad']]

    print("-> Iniciando Spark para estandarizar formato...")
    spark = (
        SparkSession.builder
        .appName("Plata_Festivos")
        .master("local[*]")
        .getOrCreate()
    )
    
    try:
        df_spark = spark.createDataFrame(df_final_pandas)
        
        # Formato fecha y eliminamos duplicados en caso de que coincidan dos fiestas
        df_limpio = df_spark.withColumn("fecha", to_date(col("fecha_cruda"))) \
                            .drop("fecha_cruda") \
                            .dropDuplicates(["fecha"])

        print(f"-> Guardando en '{CARPETA_PLATA}'...")
        if os.path.exists(CARPETA_PLATA):
            shutil.rmtree(CARPETA_PLATA)
            
        df_limpio.coalesce(1).write.mode("overwrite").parquet(CARPETA_PLATA)
        print(f"¡Éxito! Festivos procesados: {df_limpio.count()} días marcados.")
        
        df_limpio.show(5, truncate=False)

    except Exception as e:
        print(f"Error crítico en Spark: {e}")
    finally:
        spark.stop()

def ejecutar_plata_festivos():
    print("=== INICIANDO CAPA PLATA: FESTIVOS (VERSIÓN DEFINITIVA) ===")
    preparar_carpetas()
    descargar_bronze()
    transformar_y_guardar()

if __name__ == "__main__":
    ejecutar_plata_festivos()

=== INICIANDO CAPA PLATA: FESTIVOS (VERSIÓN DEFINITIVA) ===
-> Descargando JSONs de Festivos desde S3 (Modo Resumible)...
-> Descarga lista. Nuevos: 14 | En caché: 0
-> Leyendo JSONs y aplanando datos con Pandas...
-> Aplicando filtros de negocio (Solo Festivos Reales)...
-> Iniciando Spark para estandarizar formato...
-> Guardando en 'plata/festivos'...


¡Éxito! Festivos procesados: 497 días marcados.
+---------------------------+----------+
|festividad                 |fecha     |
+---------------------------+----------+
|New Year's Day             |2013-01-01|
|Epiphany                   |2013-01-06|
|Epiphany Observed          |2013-01-07|
|Day of Andalucía           |2013-02-28|
|Day of the Balearic Islands|2013-03-01|
+---------------------------+----------+
only showing top 5 rows



#### 2.3.1 Subida a S3

In [12]:
import os
import boto3

# --- CONFIGURACIÓN ---
BUCKET_NAME = "cag-proyecto-demanda-electrica-bd"
CARPETA_LOCAL_FESTIVOS = "plata/festivos"
PREFIX_S3 = "plata/festivos" 

s3 = boto3.client('s3')

def subir_festivos_s3():
    print("=== INICIANDO SUBIDA DE FESTIVOS A S3 ===")
    archivos_subidos = 0
    
    # Escaneamos la carpeta local donde Spark guardó el Parquet
    for root, dirs, files in os.walk(CARPETA_LOCAL_FESTIVOS):
        for file in files:
            # Filtramos para coger solo el archivo de datos real (ignoramos _SUCCESS, crc, etc.)
            if file.endswith(".parquet") and not file.startswith("."):
                ruta_local = os.path.join(root, file)
                
                # Construimos la ruta en S3 forzando el nombre limpio y estático
                ruta_s3 = os.path.join(PREFIX_S3, "festivos_limpios.parquet").replace("\\", "/")
                
                print(f"-> Subiendo {file} a S3 como '{ruta_s3}'...")
                s3.upload_file(ruta_local, BUCKET_NAME, ruta_s3)
                archivos_subidos += 1
                
    if archivos_subidos > 0:
        print(f"¡Éxito! El tablón de festivos ya está seguro en S3.")
    else:
        print("No se encontró ningún archivo Parquet válido para subir.")

if __name__ == "__main__":
    subir_festivos_s3()

=== INICIANDO SUBIDA DE FESTIVOS A S3 ===
-> Subiendo part-00000-428d78ef-b929-47be-8e8a-141ebdb5686d-c000.snappy.parquet a S3 como 'plata/festivos/festivos_limpios.parquet'...
¡Éxito! El tablón de festivos ya está seguro en S3.


## 3. Capa oro

In [14]:
import os
import sys
import shutil
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, dayofweek, month, year, round, quarter, lag, lit
from pyspark.sql.window import Window

# --- 1. CONFIGURACIÓN DE ENTORNO ---
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# --- 2. CONFIGURACIÓN DE RUTAS ---
BUCKET_NAME = "cag-proyecto-demanda-electrica-bd"

# Rutas en S3 de la Capa Plata
S3_PLATA_DEMANDA = "plata/demanda/demanda_limpia.parquet" 
S3_PLATA_CLIMA = "plata/clima/clima_limpio.parquet"
S3_PLATA_FESTIVOS = "plata/festivos/festivos_limpios.parquet"

# Rutas Locales
CARPETA_DESCARGAS = "descargas_oro_temp"
CARPETA_ORO_LOCAL = "oro/dataset_predictivo"
PREFIX_S3_ORO = "oro/dataset_predictivo.parquet"

s3 = boto3.client('s3')

def preparar_entorno():
    if os.path.exists(CARPETA_DESCARGAS):
        shutil.rmtree(CARPETA_DESCARGAS)
    os.makedirs(CARPETA_DESCARGAS, exist_ok=True)

def descargar_plata():
    print("-> Descargando las 3 fuentes limpias desde la Capa Plata en S3...")
    archivos = [
        (S3_PLATA_DEMANDA, "demanda.parquet"),
        (S3_PLATA_CLIMA, "clima.parquet"),
        (S3_PLATA_FESTIVOS, "festivos.parquet")
    ]
    
    for ruta_s3, nombre_local in archivos:
        ruta_destino = os.path.join(CARPETA_DESCARGAS, nombre_local)
        try:
            s3.download_file(BUCKET_NAME, ruta_s3, ruta_destino)
            print(f"   [OK] {nombre_local} descargado.")
        except Exception as e:
            print(f"   [ERROR] No se pudo descargar {ruta_s3}: {e}")
            sys.exit(1) # Si falta una pata, paramos el proceso

def crear_capa_oro():
    print("-> Levantando Spark para el cruce final...")
    spark = (
        SparkSession.builder
        .appName("Capa_Oro")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .getOrCreate()
    )
    
    try:
        # 1. Leer las tres tablas y renombrar la variable objetivo
        df_demanda = spark.read.parquet(os.path.join(CARPETA_DESCARGAS, "demanda.parquet")) \
                          .withColumnRenamed("megavatios_dia", "demanda")
                          
        df_clima = spark.read.parquet(os.path.join(CARPETA_DESCARGAS, "clima.parquet"))
        df_festivos = spark.read.parquet(os.path.join(CARPETA_DESCARGAS, "festivos.parquet"))

        print("-> Realizando LEFT JOIN y Feature Engineering...")
        
        # 2. Cruce Maestro
        df_oro = df_demanda.join(df_clima, "fecha", "left") \
                           .join(df_festivos, "fecha", "left")
        
        # Definimos la ventana temporal para poder calcular el consumo de hace 7 días
        # Le decimos explícitamente que todo va a una misma partición
        windowSpec = Window.partitionBy(lit(1)).orderBy("fecha")
        
        # 3. Feature Engineering (Creación de TODAS las variables del Diccionario de Datos)
        df_oro = df_oro \
            .withColumn("es_festivo", when(col("festividad").isNotNull(), 1).otherwise(0)) \
            .withColumn("festividad", when(col("festividad").isNull(), "Laborable/Normal").otherwise(col("festividad"))) \
            .withColumn("dia_semana", dayofweek(col("fecha"))) \
            .withColumn("mes", month(col("fecha"))) \
            .withColumn("anio", year(col("fecha"))) \
            .withColumn("trimestre", quarter(col("fecha"))) \
            .withColumn("es_fin_de_semana", when(col("dia_semana").isin(1, 7), 1).otherwise(0)) \
            .withColumn("rango_termico", round(col("temp_max_media") - col("temp_min_media"), 2)) \
            .withColumn("demanda_lag_7", lag("demanda", 7).over(windowSpec))
            
        # 4. Limpieza final: Ordenamos y quitamos la primera semana que genera nulos en el lag_7
        df_oro = df_oro.orderBy("fecha").filter(col("demanda_lag_7").isNotNull())

        print(f"-> Guardando Dataset Predictivo en '{CARPETA_ORO_LOCAL}'...")
        if os.path.exists(CARPETA_ORO_LOCAL):
            shutil.rmtree(CARPETA_ORO_LOCAL)
            
        df_oro.coalesce(1).write.mode("overwrite").parquet(CARPETA_ORO_LOCAL)
        
        total_filas = df_oro.count()
        print(f"¡Capa Oro finalizada! Tablón maestro generado con {total_filas} días.")
        
        # Mostramos esquema final y una muestra de los datos incluyendo las nuevas variables
        df_oro.printSchema()
        # Muestra todas las columnas, 5 filas, y sin recortar el texto
        df_oro.show(5, truncate=False)
        
    except Exception as e:
        print(f"Error crítico en el cruce de Spark: {e}")
    finally:
        spark.stop()

def subir_oro_s3():
    print("=== SUBIENDO DATASET PREDICTIVO A S3 ===")
    # Buscamos el archivo generado y lo subimos con nombre limpio
    for file in os.listdir(CARPETA_ORO_LOCAL):
        if file.endswith(".parquet") and not file.startswith("."):
            ruta_local = os.path.join(CARPETA_ORO_LOCAL, file)
            print(f"-> Subiendo a S3 como '{PREFIX_S3_ORO}'...")
            s3.upload_file(ruta_local, BUCKET_NAME, PREFIX_S3_ORO)
            print("¡PROYECTO DE INGESTA COMPLETADO!")
            break

def ejecutar_capa_oro():
    print("=== INICIANDO CAPA ORO: DATASET MAESTRO ===")
    preparar_entorno()
    descargar_plata()
    crear_capa_oro()
    subir_oro_s3()

if __name__ == "__main__":
    ejecutar_capa_oro()

=== INICIANDO CAPA ORO: DATASET MAESTRO ===
-> Descargando las 3 fuentes limpias desde la Capa Plata en S3...
   [OK] demanda.parquet descargado.
   [OK] clima.parquet descargado.
   [OK] festivos.parquet descargado.
-> Levantando Spark para el cruce final...
-> Realizando LEFT JOIN y Feature Engineering...
-> Guardando Dataset Predictivo en 'oro/dataset_predictivo'...
¡Capa Oro finalizada! Tablón maestro generado con 4880 días.
root
 |-- fecha: date (nullable = true)
 |-- demanda: float (nullable = true)
 |-- temp_max_media: double (nullable = true)
 |-- temp_min_media: double (nullable = true)
 |-- temp_media_nacional: double (nullable = true)
 |-- precipitacion_media: double (nullable = true)
 |-- festividad: string (nullable = true)
 |-- es_festivo: integer (nullable = false)
 |-- dia_semana: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- anio: integer (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- es_fin_de_semana: integer (nullable = false)
 